# EasyRAG Normalization Before Embedding

## What you will do

- compare raw and normalized chunk text before embedding
- inspect deterministic embedding vectors for both versions
- measure a simple query-to-chunk similarity score before and after cleanup

## Why this matters

EasyRAG normalizes text before indexing on purpose. Embeddings are only as stable as the strings you feed them. Broken whitespace and hidden bytes can change token boundaries in ways that are hard to notice later.


## Source code anchors

- `easyrag.rag.indexing.normalization.normalize_document_text`
- `easyrag.rag.orchestrator.EasyRAG.prepare_documents`
- `tests.test_rag_retriever._stub_embedding`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

import math

from easyrag.rag.indexing.normalization import normalize_document_text

def cosine_similarity(left: list[float], right: list[float]) -> float:
    numerator = sum(a * b for a, b in zip(left, right, strict=False))
    left_norm = math.sqrt(sum(a * a for a in left))
    right_norm = math.sqrt(sum(b * b for b in right))
    if left_norm == 0 or right_norm == 0:
        return 0.0
    return numerator / (left_norm * right_norm)


## Deterministic path

The raw text below is intentionally noisy. A null byte splits the word `retrieval`, and repeated line breaks make the chunk harder to inspect. The deterministic embedding stub is simple enough that you can see exactly why normalization helps.


In [ ]:
raw_text = "# Retrieval\nEasyRAG keeps grounded retrie\x00val stable.\n\n\nQuery rewrite improves routing.  \n"
normalized_text = normalize_document_text(raw_text)
query_text = "How does grounded retrieval use query rewrite?"

raw_vector, normalized_vector, query_vector = _stub_embedding([raw_text, normalized_text, query_text])
comparison = {
    "raw_text": raw_text,
    "normalized_text": normalized_text,
    "raw_vector": raw_vector,
    "normalized_vector": normalized_vector,
    "query_vector": query_vector,
    "raw_query_similarity": round(cosine_similarity(raw_vector, query_vector), 4),
    "normalized_query_similarity": round(cosine_similarity(normalized_vector, query_vector), 4),
}
print(json.dumps(comparison, indent=2, sort_keys=True))


## Output inspection

The cleaned text is still semantically the same sentence, but the keyword boundary is now recoverable. That means the embedding input is more faithful to what a retriever is supposed to rank.


In [ ]:
prepared_document = EasyRAG.prepare_documents(
    [raw_text],
    ids=["doc::retrieval"],
    file_paths=["docs/retrieval.md"],
)[0]

print(prepared_document.page_content)
print(json.dumps(prepared_document.metadata, indent=2, sort_keys=True))


## Provider-backed path

If the environment is configured, the next cell embeds the raw and normalized text with the repo's default embedding function. The goal is not to claim a universal numeric improvement. The goal is to inspect whether the provider sees the same cleaned boundary that the deterministic path highlighted.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_rag = EasyRAG(workspace="provider-normalization-before-embedding")
    try:
        provider_vectors = provider_rag.embedding_func([raw_text, normalized_text, query_text])
        provider_summary = {
            "vector_lengths": [len(vector) for vector in provider_vectors],
            "raw_first_values": provider_vectors[0][:5],
            "normalized_first_values": provider_vectors[1][:5],
        }
        print(json.dumps(provider_summary, indent=2, default=str))
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")


## What changed and why

The important change is not cosmetic cleanup. It is boundary recovery. Once the hidden byte disappears, `retrieval` becomes a stable token again, the vector changes, and the query-to-chunk similarity improves. EasyRAG performs this cleanup before indexing so those small string-level problems do not silently degrade dense retrieval.


## Next steps

- Continue with [03_07_build_index_pipeline.ipynb](03_07_build_index_pipeline.ipynb) to see how normalized chunks move into summaries, vectors, graph records, and storage writes.
- Read [03-indexing-overview.md](../../docs/03-indexing-overview.md) for the stage-level explanation of chunking, embedding, and index assembly.
- Revisit [03_01_chunking_principles.ipynb](03_01_chunking_principles.ipynb) if you want to contrast boundary choice with embedding-input quality.
